In [ ]:

# FEDERATED TRAINING — 100% FaceForensics++
# Xception + RGB-FFT + Class-Aware SFIAD

import os, glob, copy, random, time
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler
from torch import amp

from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
import timm

# ---------------- CONFIG ----------------
ROOT = r"E:\SFIAD_Project\ffpp_fused_full_final\clients_fused_tt"
CLIENTS = ["client1", "client2", "client3", "client4"]

USE_FRACTION = 1.0
BATCH_SIZE = 16
NUM_ROUNDS = 12
LR = 1e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

SAVE_PATH = r"E:\SFIAD_Project\models\sfiad_xception_fedavg_100pct1.pth"
# ---------------------------------------

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ---------------- DATASET ----------------
class FFPPDataset(Dataset):
    def __init__(self, root, fraction=1.0):
        self.samples = []
        for label, cls in enumerate(["real", "fake"]):
            files = glob.glob(os.path.join(root, cls, "*.npy"))
            random.shuffle(files)
            files = files[:int(len(files) * fraction)]
            for f in files:
                self.samples.append((f, label))
        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        x = np.load(path)
        if x.shape[0] != 4:
            x = np.transpose(x, (2, 0, 1))
        return torch.tensor(x, dtype=torch.float32), label
# ----------------------------------------

# ---------------- MODEL ------------------
class AAMSoftmax(nn.Module):
    def __init__(self, feat_dim, n_classes=2):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_classes, feat_dim))

class XceptionSFIAD(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "xception",
            pretrained=True,
            num_classes=0,
            in_chans=4
        )
        feat_dim = self.backbone.num_features
        self.cls_loss = AAMSoftmax(feat_dim, 2)

    def forward(self, x):
        return self.backbone(x)
# ----------------------------------------

# ---------------- LOSS -------------------
def sfiad_loss(feats, labels, cls_layer, m_fake=0.2):
    logits = F.linear(F.normalize(feats), F.normalize(cls_layer.weight))
    cls_loss = F.cross_entropy(logits, labels)

    half = feats.shape[1] // 2
    rgb_f = feats[:, :half]
    fft_f = feats[:, half:]

    real_mask = labels == 0
    fake_mask = labels == 1

    cons_loss = torch.tensor(0.0, device=feats.device)

    if real_mask.any():
        cons_loss += F.mse_loss(rgb_f[real_mask], fft_f[real_mask])

    if fake_mask.any():
        dist = F.pairwise_distance(rgb_f[fake_mask], fft_f[fake_mask])
        cons_loss += torch.mean(F.relu(m_fake - dist))

    return cls_loss + cons_loss
# ----------------------------------------

# ---------------- TRAIN / EVAL FUNCTION ----------
def run_epoch(model, loader, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_probs, all_labels = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        with amp.autocast(device_type="cuda"):
            feats = model(x)
            loss = sfiad_loss(feats, y, model.cls_loss)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item()

        with torch.no_grad():
            feats_fp32 = feats.float()
            weights_fp32 = model.cls_loss.weight.float()

            logits = F.linear(
                F.normalize(feats_fp32),
                F.normalize(weights_fp32)
            )
            probs = torch.softmax(logits, dim=1)[:, 1]

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    auc = roc_auc_score(all_labels, all_probs)
    ap  = average_precision_score(all_labels, all_probs)
    acc = accuracy_score(all_labels, np.array(all_probs) > 0.5)

    return total_loss / max(1, len(loader)), auc, ap, acc
# ----------------------------------------

# ---------------- FEDERATED TRAINING ----------------
global_model = XceptionSFIAD().to(DEVICE)
scaler = GradScaler()

best_global_auc = 0.0
best_round = -1

start_time = time.time()

for rnd in range(NUM_ROUNDS):
    print("\nFederated Round {}/{}".format(rnd + 1, NUM_ROUNDS))
    client_states = []

    for cname in CLIENTS:
        print(" Training", cname)

        ds = FFPPDataset(os.path.join(ROOT, cname, "train"), USE_FRACTION)
        labels = [y for _, y in ds.samples]
        weights = [1.0 / Counter(labels)[y] for y in labels]

        loader = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            sampler=WeightedRandomSampler(weights, len(weights)),
            num_workers=0,
            pin_memory=True
        )

        local_model = copy.deepcopy(global_model)
        optimizer = torch.optim.Adam(local_model.parameters(), lr=LR)

        loss, auc, ap, acc = run_epoch(local_model, loader, optimizer, scaler)
        print("  Client Loss {:.4f} | AUC {:.4f} | AP {:.4f} | Acc {:.4f}".format(
            loss, auc, ap, acc
        ))

        client_states.append(copy.deepcopy(local_model.state_dict()))

        del local_model
        torch.cuda.empty_cache()

    # -------- FedAvg --------
    new_state = copy.deepcopy(global_model.state_dict())
    for k in new_state:
        new_state[k] = torch.stack(
            [cs[k].float() for cs in client_states], 0
        ).mean(0)

    global_model.load_state_dict(new_state)

    # -------- GLOBAL MODEL EVALUATION --------
    print(" Evaluating Global Model")

    all_ds = []
    for cname in CLIENTS:
        all_ds.extend(
            FFPPDataset(os.path.join(ROOT, cname, "train"), USE_FRACTION).samples
        )

    class GlobalDataset(Dataset):
        def __init__(self, samples):
            self.samples = samples
        def __len__(self):
            return len(self.samples)
        def __getitem__(self, idx):
            path, label = self.samples[idx]
            x = np.load(path)
            if x.shape[0] != 4:
                x = np.transpose(x, (2, 0, 1))
            return torch.tensor(x, dtype=torch.float32), label

    global_loader = DataLoader(
        GlobalDataset(all_ds),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    g_loss, g_auc, g_ap, g_acc = run_epoch(global_model, global_loader)
    print("  Global Loss {:.4f} | AUC {:.4f} | AP {:.4f} | Acc {:.4f}".format(
        g_loss, g_auc, g_ap, g_acc
    ))

    # -------- SAVE BEST GLOBAL MODEL --------
    if g_auc > best_global_auc:
        best_global_auc = g_auc
        best_round = rnd + 1
        torch.save(global_model.state_dict(), SAVE_PATH)
        print("  Best global model updated at round", best_round)

    torch.cuda.empty_cache()

elapsed = (time.time() - start_time) / 3600
print("\nTotal training time {:.2f} hours".format(elapsed))

print("\nBest Global Model Summary")
print("Best AUC   :", best_global_auc)
print("Best Round :", best_round)
print("Model Path :", SAVE_PATH)
